# ⚽ Mission 12: Soccer Vision Lab — Build an AI Tactical Analyst

## 📖 Mission Story
Welcome back! 🎉

During the first eleven missions, you gradually learned how to become an AI-powered soccer coach. You started with Python programming, then learned how to organize data with Pandas, visualize player statistics, build professional Streamlit dashboards, and even consult Gemini AI to answer tactical questions.

But there has always been one important limitation: every dashboard you built assumed that someone had already collected the player statistics. Professional soccer clubs don't work that way. Instead of receiving ready-made numbers, they begin with a match video!

Today, you are going to build your own sports analytics system. By the end of this mission, your Soccer AI Coach will no longer depend on manually entered statistics—it will create its own statistics directly from match footage. Welcome to the world of **Computer Vision**!

---

## 🎯 Learning Objectives
* ✅ Explain how computers represent videos as sequences of digital images (frames).
* ✅ Read soccer videos frame-by-frame using OpenCV (`cv2`).
* ✅ Upload video files inside Streamlit using `st.file_uploader()`.
* ✅ Detect soccer players inside video frames using YOLO (`ultralytics`).
* ✅ Understand the key difference between **Object Detection** and **Object Tracking**.
* ✅ Convert camera pixel coordinates into real soccer pitch coordinates ($120 \times 80$ yards).
* ✅ Generate professional tactical heatmaps using `mplsoccer`.
* ✅ Ask Gemini AI to analyze player movement as an elite UEFA Pro Analyst.
* ✅ Combine Computer Vision, Data Science, Visualization, and AI into one complete application.

---

## 🟢 Phase 1: Understanding Soccer Heatmaps

Before we teach the computer how to collect movement from video, we first need to learn how to visualize movement on a soccer pitch.

In modern sports analytics, a **Heatmap** shows where a player spends most of their time during a match. Every time a player touches the ball or moves into space, we record their position as an $(X, Y)$ coordinate:
- **X Coordinate (0 to 120 yards):** The length of the pitch (0 = own goal line, 120 = opponent's goal line).
- **Y Coordinate (0 to 80 yards):** The width of the pitch (0 = left touchline, 80 = right touchline).

Let's write a program using `mplsoccer` and `matplotlib` to plot pitch movement:

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import numpy as np

# 1. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# 2. Simulate positional data
player = "Artin"
position = "Right Defender"

if position == "Left Winger":
    x_coordinates = [10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105]
    y_coordinates = [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]
elif position == "Striker":
    x_coordinates = [70,75,80,85,90,95,100,102,105,108,110,112,115,108,104,100,95,90,88,110]
    y_coordinates = [35,38,40,42,40,38,35,37,40,42,39,36,40,45,48,50,45,42,38,35]
elif position == "Midfielder":
    x_coordinates = [35,40,45,50,55,60,65,70,60,55,50,45,40,55,65,75,70,60,50,45]
    y_coordinates = [25,30,35,40,45,40,35,30,25,20,25,30,35,45,50,45,40,35,30,25]
elif position == "Right Defender":
    x_coordinates = [15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110]
    y_coordinates = [70,72,68,70,72,74,76,74,72,70,68,70,72,74,76,72,68,65,60,55]

# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw individual touch points on top
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(f"{player}'s Soccer Heatmap - {position}", fontsize=18, fontweight='bold', pad=15)
plt.show()

## 📊 Phase 2: From Manual Coordinates to Real Player Data

In real applications, coordinates are loaded dynamically from files. We manage movement tracking logs using **Pandas DataFrames** and store them into `.csv` files.

In [ ]:
%%writefile phase2_csv_loader.py
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Create synthetic player tracking dataset
data = {
    'frame': list(range(1, 11)),
    'x': [18, 20, 21, 25, 30, 42, 55, 68, 72, 85],
    'y': [52, 53, 54, 50, 48, 45, 40, 38, 35, 30]
}

# Save to CSV
df = pd.DataFrame(data)
df.to_csv("player_movement.csv", index=False)
print("✅ CSV file 'player_movement.csv' created successfully!")

# Load CSV and render Kernel Density Estimate (KDE) smooth heatmap
movement_df = pd.read_csv("player_movement.csv")
print("Loaded Data:")
print(movement_df.head())

pitch = Pitch(pitch_type='statsbomb', pitch_color='#101010', line_color='#888888')
fig, ax = pitch.draw(figsize=(10, 7))
pitch.kdeplot(movement_df['x'], movement_df['y'], ax=ax, cmap='magma', fill=True, alpha=0.65)
plt.title("Artin Movement Trajectory (KDE Density)")
plt.savefig("phase2_heatmap.png", bbox_inches='tight')
print("✅ Saved plot as 'phase2_heatmap.png'!")

## 🎥 Phase 3: Reading Soccer Videos with OpenCV

A video is simply a fast sequence of still images called **frames**. OpenCV (`cv2`) allows us to open videos, extract metadata, and process frames sequentially.

In [ ]:
%%writefile phase3_opencv_intro.py
import cv2

def inspect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Cannot open video file: {video_path}")
        return
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    print("📹 Video Metadata Summary")
    print("------------------------")
    print(f"Resolution : {width} x {height} pixels")
    print(f"FPS        : {fps:.2f} frames/sec")
    print(f"Frames     : {frame_count}")
    print(f"Duration   : {duration:.2f} seconds")
    
    success, frame = cap.read()
    if success:
        cv2.imwrite("first_frame.jpg", frame)
        print("✅ Saved first frame as 'first_frame.jpg'")
    
    cap.release()

## 💻 Phase 4: Building a Soccer Video Upload Interface with Streamlit

Streamlit's `st.file_uploader()` with Python's `tempfile` module allows dynamic video ingestion for computer vision pipelines.

In [ ]:
%%writefile phase4_video_upload.py
import streamlit as st
import cv2
import tempfile

st.title("⚽ Soccer Video Analyzer")
uploaded_file = st.file_uploader("Upload Match Video", type=["mp4", "mov", "avi"])

if uploaded_file is not None:
    st.video(uploaded_file)
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    video_path = tfile.name
    
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    st.subheader("📹 Technical Video Metrics")
    st.write(f"**Resolution:** {width} x {height} px")
    st.write(f"**FPS:** {fps:.2f}")
    st.write(f"**Total Frames:** {frame_count}")
    if fps > 0:
        st.write(f"**Duration:** {frame_count / fps:.2f} seconds")
    cap.release()

## 🤖 Phase 5: YOLO — Detection & Object Tracking

- **Object Detection:** Locates object boundaries frame-by-frame independently.
- **Object Tracking:** Tracks identities by assigning persistent **Track IDs** across sequential frames using models like YOLOv8.

In [ ]:
%%writefile phase5_yolo_demo.py
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("✅ YOLOv8 model loaded successfully!")

## 🟢 Phase 6: Coordinate Mapping — From Pixel Space to Pitch Space

Converting pixel bounding coordinates to soccer pitch dimensions ($120 \times 80$ yards).

In [ ]:
%%writefile tracking_utils.py
def clamp(value, minimum, maximum):
    return max(minimum, min(value, maximum))

def map_pixels_to_pitch(pixel_x, pixel_y, frame_width=1280, frame_height=720, pitch_length=120.0, pitch_width=80.0):
    raw_pitch_x = (pixel_x / frame_width) * pitch_length
    raw_pitch_y = (pixel_y / frame_height) * pitch_width
    
    pitch_x = clamp(raw_pitch_x, 0.0, pitch_length)
    pitch_y = clamp(raw_pitch_y, 0.0, pitch_width)
    
    return round(pitch_x, 2), round(pitch_y, 2)

## 🏆 Final Boss Project: From Video to AI Coaching Report

### 📖 Mission Story
Congratulations, Coach Artin! 🎉⚽

You have now learned how professional soccer analysts transform raw match footage into meaningful insights.
During the previous phases, you built the individual pieces:

| Mission | Skill | What You Built |
|---|---|---|
| Mission 2 | Data Basics | Loading and exploring soccer data |
| Mission 7 | Sports Data | Soccer analytics |
| Mission 8 | Data Viz | Visualization |
| Mission 9 | Web Apps | Streamlit dashboards |
| Mission 10 | Generative AI | Gemini AI coaching assistant |
| Mission 11 | Architecture | Professional multi-page applications |
| Mission 12 (Phases 1-7) | Computer Vision | Soccer vision analytics |

Now it is time to combine everything into one professional AI-powered soccer analysis application. Professional clubs do not have separate tools for every task. They use one platform:

```
              Soccer Video
                    |
                    ↓
              Computer Vision
                    |
                    ↓
          Player Movement Data
                    |
                    ↓
       Tactical Visualization
                    |
                    ↓
             AI Coach Report
```

Today you will build: **⚽ Soccer Vision AI Tactical Analyst**

--- 

### 🎯 Final Learning Objectives
By the end of this project, you will be able to:
* ✅ Build a complete Streamlit AI application
* ✅ Upload and process soccer videos
* ✅ Extract frames using OpenCV
* ✅ Detect players using YOLO
* ✅ Track player movement
* ✅ Convert video coordinates into soccer coordinates
* ✅ Generate professional heatmaps
* ✅ Calculate movement statistics
* ✅ Build dynamic Gemini prompts
* ✅ Create AI-generated tactical reports

--- 

### 🧩 Application Architecture
Your final application will have this structure:

```
Soccer_Vision_Lab/
│
├── app.py
│
├── utils/
│   ├── video_utils.py
│   ├── tracking_utils.py
│   └── tactical_utils.py
│
├── models/
│   └── yolov8n.pt
│
├── outputs/
│   └── heatmaps/
│
└── requirements.txt
```

#### Why split the code?
Remember Mission 4: You created `coach_toolbox.py` instead of writing everything in one notebook. Professional developers do the same. Large applications are divided into smaller tools.

### Phase 8.1 — Create Utility Functions

#### Step 1: Create a toolbox for soccer vision
Create `vision_toolbox.py`:

In [ ]:
%%writefile vision_toolbox.py
import pandas as pd

def calculate_distance(points):
    """
    Calculate approximate player movement distance
    """
    distance = 0
    for i in range(len(points)-1):
        x1, y1 = points[i]
        x2, y2 = points[i+1]
        distance += ((x2-x1)**2 + (y2-y1)**2)**0.5
    return round(distance, 2)

def classify_player_style(avg_x):
    if avg_x > 90:
        return "Highly attacking player"
    elif avg_x < 40:
        return "Defensive player"
    else:
        return "Balanced player"

#### 🧪 Exercise 1
Test your toolbox by executing the code block below:

In [ ]:
import vision_toolbox as vt

movement = [
    (20, 40),
    (30, 45),
    (40, 50)
]

print(vt.calculate_distance(movement))
# Expected Output: 28.28

> **Reflection Question:** Why did we create a function?  
> **Answer:** Because `calculate_distance()` can now be reused in our Streamlit app, Jupyter notebooks, or future sports analytics projects!

### Phase 8.2 — Build Streamlit Interface

Let's create the entry point `app.py` and configure the initial control layout:

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from ultralytics import YOLO
import google.generativeai as genai
import vision_toolbox as vt
from tracking_utils import map_pixels_to_pitch

st.set_page_config(
    page_title="Soccer Vision Lab",
    page_icon="⚽",
    layout="wide"
)

st.title("⚽ Soccer Vision AI Tactical Analyst")
st.write("Upload a soccer video and receive professional tactical analysis.")

# Phase 8.2 Sidebar Control Panel
st.sidebar.title("Coach Control Panel")
player_name = st.sidebar.text_input("Player Name", value="Artin")

#### ✏️ Exercise 2
Your interface control panel layout is now configured as:
```
----------------------
Coach Control Panel

Player Name:
[ Artin ]
----------------------
Soccer Vision AI Tactical Analyst
```

### Phase 8.3 & 8.4 — Upload Soccer Video & Information Panel

Next, we expand `app.py` to allow file uploads using `st.file_uploader()` and display video metrics using `st.metric()`.

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from ultralytics import YOLO
import google.generativeai as genai
import vision_toolbox as vt
from tracking_utils import map_pixels_to_pitch

st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")
st.title("⚽ Soccer Vision AI Tactical Analyst")
st.write("Upload a soccer video and receive professional tactical analysis.")

st.sidebar.title("Coach Control Panel")
player_name = st.sidebar.text_input("Player Name", value="Artin")
target_player_id = st.sidebar.number_input("Target Track ID", min_value=1, value=1)

uploaded_video = st.file_uploader("Upload Match Video", type=["mp4"])

if uploaded_video:
    st.video(uploaded_video)
    
    # Phase 8.4: Video Information Panel
    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(uploaded_video.read())
    
    cap = cv2.VideoCapture(temp_file.name)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    col1, col2 = st.columns(2)
    col1.metric("Frames", frames)
    col2.metric("FPS", round(fps, 2))
    cap.release()

#### ✏️ Exercise 3
Now Coach Artin can upload `Artin_match.mp4` and watch it directly inside the app while viewing total frame counts and FPS!

### Phase 8.5, 8.6 & 8.7 — YOLO Player Tracking, Data Table & Heatmap Generation

We integrate YOLOv8 tracking to extract frame bounding boxes, map them onto pitch space, calculate movement dataframes, and render heatmaps with `mplsoccer`.

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from ultralytics import YOLO
import google.generativeai as genai
import vision_toolbox as vt
from tracking_utils import map_pixels_to_pitch

st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")
st.title("⚽ Soccer Vision AI Tactical Analyst")
st.write("Upload a soccer video and receive professional tactical analysis.")

st.sidebar.title("Coach Control Panel")
player_name = st.sidebar.text_input("Player Name", value="Artin")
target_player_id = st.sidebar.number_input("Target Track ID", min_value=1, value=1)

uploaded_video = st.file_uploader("Upload Match Video", type=["mp4"])

if uploaded_video:
    st.video(uploaded_video)
    
    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(uploaded_video.read())
    
    cap = cv2.VideoCapture(temp_file.name)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    col1, col2 = st.columns(2)
    col1.metric("Frames", frames)
    col2.metric("FPS", round(fps, 2))
    
    # Phase 8.5: YOLO Tracking
    model = YOLO("yolov8n.pt")
    tracking_records = []
    points_list = []
    frame_idx = 0
    
    st.info("Processing video tracking...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= 100:
            break
        frame_idx += 1
        
        results = model.track(frame, persist=True, verbose=False)[0]
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()
            
            for box, track_id, cls in zip(boxes, track_ids, clss):
                if int(cls) == 0 and int(track_id) == target_player_id:
                    cx = (box[0] + box[2]) / 2.0
                    cy = box[3]
                    px, py = map_pixels_to_pitch(cx, cy, frame_w, frame_h)
                    tracking_records.append({"frame": frame_idx, "x": px, "y": py})
                    points_list.append((px, py))
    cap.release()
    
    if tracking_records:
        movement_df = pd.DataFrame(tracking_records)
        
        # Phase 8.6: Tactical Dashboard Table
        st.subheader("📊 Movement Tracking Log")
        st.dataframe(movement_df)
        
        # Phase 8.7: Pitch Heatmap Rendering
        st.subheader("📍 Tactical Heatmap")
        pitch = Pitch(pitch_type="statsbomb", pitch_color="#aabb97", line_color="white")
        fig, ax = pitch.draw()
        pitch.kdeplot(movement_df["x"], movement_df["y"], ax=ax, fill=True, cmap="Reds", alpha=0.6)
        st.pyplot(fig)

#### ✏️ Exercise 4
Inspecting the raw tracking results shows how pixel boxes and IDs are tracked continuously through the execution pipeline!

### Phase 8.8 — Gemini Tactical Coach & Final Challenge

Here is the complete application containing **Phase 8.8** and all requirements for the **🏆 Coach Artin Pro Mode Challenge** (Position selectors, distance metrics, tailored tactical questions, and direct report downloads).

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import tempfile
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from ultralytics import YOLO
import google.generativeai as genai
import vision_toolbox as vt
from tracking_utils import map_pixels_to_pitch

st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")
st.title("⚽ Soccer Vision AI Tactical Analyst")
st.write("Upload a soccer video and receive professional tactical analysis.")

# Sidebar Control Panel
st.sidebar.title("Coach Control Panel")
player_name = st.sidebar.text_input("Player Name", value="Artin")
target_player_id = st.sidebar.number_input("Target Track ID", min_value=1, value=1)

# 🏆 Pro Mode Extension 1: Player Position Selector
player_position = st.sidebar.selectbox("Player Position", ["Forward", "Midfielder", "Defender"])

uploaded_video = st.file_uploader("Upload Match Video", type=["mp4"])

if uploaded_video:
    st.video(uploaded_video)
    
    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(uploaded_video.read())
    
    cap = cv2.VideoCapture(temp_file.name)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    col1, col2 = st.columns(2)
    col1.metric("Frames", frames)
    col2.metric("FPS", round(fps, 2))
    
    model = YOLO("yolov8n.pt")
    tracking_records = []
    points_list = []
    frame_idx = 0
    
    st.info("Processing video tracking...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= 100:
            break
        frame_idx += 1
        
        results = model.track(frame, persist=True, verbose=False)[0]
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()
            
            for box, track_id, cls in zip(boxes, track_ids, clss):
                if int(cls) == 0 and int(track_id) == target_player_id:
                    cx = (box[0] + box[2]) / 2.0
                    cy = box[3]
                    px, py = map_pixels_to_pitch(cx, cy, frame_w, frame_h)
                    tracking_records.append({"frame": frame_idx, "x": px, "y": py})
                    points_list.append((px, py))
    cap.release()
    
    if tracking_records:
        movement_df = pd.DataFrame(tracking_records)
        
        st.subheader("📊 Movement Tracking Log")
        st.dataframe(movement_df)
        
        st.subheader("📍 Tactical Heatmap")
        pitch = Pitch(pitch_type="statsbomb", pitch_color="#aabb97", line_color="white")
        fig, ax = pitch.draw()
        pitch.kdeplot(movement_df["x"], movement_df["y"], ax=ax, fill=True, cmap="Reds", alpha=0.6)
        st.pyplot(fig)
        
        # 🏆 Pro Mode Extension 3: Performance Metrics
        st.subheader("📈 Performance Metrics")
        distance = vt.calculate_distance(points_list)
        avg_x = movement_df["x"].mean()
        avg_y = movement_df["y"].mean()
        style = vt.classify_player_style(avg_x)
        
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Distance Covered", f"{distance} yds")
        m2.metric("Avg Position X", f"{avg_x:.1f}")
        m3.metric("Avg Position Y", f"{avg_y:.1f}")
        m4.metric("Role Style", style)
        
        # Phase 8.8: Gemini Tactical Coach Integration
        st.divider()
        st.subheader("🤖 Gemini AI Tactical Coach")
        
        # 🏆 Pro Mode Extension 2: Position-tailored question
        if player_position == "Forward":
            specific_question = "Analyze my attacking movement and box entry efficiency."
        elif player_position == "Midfielder":
            specific_question = "Analyze my central coverage and ball distribution positioning."
        else:
            specific_question = "Analyze my defensive positioning and defensive line discipline."
            
        try:
            genai.configure(api_key=st.secrets["GEMINI_API_KEY"])
            coach = genai.GenerativeModel("gemini-1.5-flash")
            
            context = f"""
            Player Name: {player_name}
            Role: {player_position}
            Average X position: {avg_x:.2f}
            Average Y position: {avg_y:.2f}
            Total Distance Covered: {distance} yards
            Calculated Playing Style: {style}
            """
            
            prompt = f"""
            You are a UEFA professional tactical analyst.
            
            Analyze this player's movement.
            {context}
            
            Tactical Focus Question: {specific_question}
            
            Provide:
            1. Tactical strengths
            2. Weaknesses
            3. Direct training recommendations
            """
            
            if st.button("Generate Tactical Coaching Report"):
                with st.spinner("Analyzing tactical movement..."):
                    response = coach.generate_content(prompt)
                    report_text = response.text
                    st.write(report_text)
                    
                    # 🏆 Pro Mode Extension 4: Download Button
                    st.download_button(
                        label="📥 Download Tactical Report",
                        data=report_text,
                        file_name=f"{player_name}_Tactical_Report.txt",
                        mime="text/plain"
                    )
        except Exception as e:
            st.warning("Provide GEMINI_API_KEY in st.secrets to enable AI Tactical Analysis.")

### 🌟 Final Reflection & Mission Complete

Coach Artin, you have now built a simplified version of professional systems used by **Hudl**, **StatsBomb**, and **Wyscout**!

```
Python ➔ Data Analysis ➔ Visualization ➔ Streamlit Apps ➔ Generative AI ➔ Computer Vision ➔ AI Soccer Analyst
```

## 🚀 Mission Complete
You are no longer only analyzing soccer data. You are building AI systems for soccer intelligence! ⚽🤖  
**End of Mission 12 Final Project**